# Significance Testing via Permutation Tests

- Text level was shuffled in training w/ a different random seed for each permutation

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.utils import ALPHA_LIST
import pandas as pd
import numpy as np
import shlex
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from src.perm_res import (
    get_p_df,
    expand_alpha_rows,
    collapse_hypothesis,
    get_collapsed_null,
    get_ranking,
)

In [ ]:
search_dict = {
    "agg": ["binary", "proba-limited", "avg", "block", "max"],
    "llama_model": ["Llama-3.3-8B-Instruct", "Llama-3.3-70B-Instruct"],
    "desc": ["augment", "augment_neg", "original"],
    "seg": ["pt_noshort", "turns"],
    "alpha": ALPHA_LIST,
}
comprehensive_dir = BASE_PATH / "perm_res" / "r2_comprehensive"

# Part 1

## Aggregate r^2 values across perms

- One table per impact/num-layers combination
- Each table contains r2_S,r2_Q,r2_T for some model configuration

In [ ]:
def create_cmd(
    num_layers, import_dir, search_dict, export_path, n_threads, num_repeats
):
    cmd_str_list = [
        f"export PYTHONPATH={BASE_PATH};",
        f"export OMP_NUM_THREADS={n_threads};",
        "uv run",
        "python -m src.perm_res",
        "--num_layers",
        str(num_layers),
        "--import_dir",
        str(import_dir),
        "--export_path",
        str(export_path),
        "--search_dict",
        shlex.quote(json.dumps(search_dict)),
        "--num_repeats",
        str(num_repeats),
    ]
    return " ".join(cmd_str_list)

In [ ]:
perm_check_start = 0
perm_check_end = 1000  # not inclusive
INCLUDE_BASELINE = True

In [ ]:
full_cmd_list = []
loop_dir = BASE_PATH / "perm_res" / "preds"
swarm_dir = BASE_PATH / "swarm_dir/perm_r2"
cmd_dir = swarm_dir / "commands"
log_dir = swarm_dir / "logs"
r2_dir = BASE_PATH / "perm_res" / "r2_tables"
for impact_type in ["parent", "self"]:
    for num_layers in [2, 3]:
        swarm_path = cmd_dir / f"{impact_type}_{num_layers}.swarm"
        swarm_path.parent.mkdir(exist_ok=True, parents=True)
        if swarm_path.exists():
            swarm_path.unlink()
        cmd_list = []
        for perm_dir in loop_dir.iterdir():
            export_path = r2_dir / perm_dir.name / f"{impact_type}_{num_layers}.parquet"
            perm_name = perm_dir.name
            if perm_name == "baseline":
                if not INCLUDE_BASELINE:
                    continue
            else:
                perm_num = int(perm_name.split("_")[1].strip(""))
                if (perm_num < perm_check_start) or (perm_num >= perm_check_end):
                    continue
            print(perm_name)

            cmd_str = create_cmd(
                num_layers=num_layers,
                import_dir=perm_dir / impact_type,
                search_dict=search_dict,
                n_threads=2,
                export_path=export_path,
                num_repeats=10,
            )
            cmd_list.append(cmd_str)
        swarm_path.write_text("\n".join(cmd_list))
        full_cmd_list += cmd_list
print(f"Num commands: {len(full_cmd_list)}")

In [ ]:
pd.read_parquet(
    "/data/MLDSST/mataray/DAWBA/perm_res/preds/baseline/parent/pred_df_combined.parquet"
)

Run w/ swarm

In [ ]:
from src.swarm import run_perm_r2_swarm

run_perm_r2_swarm(
    impact_list=["parent", "self"],
    layer_list=[2, 3],
    log_dir=log_dir,
    cmd_dir=cmd_dir,
    n_batch_for_3=8,
)

Run sequentially

In [ ]:
# import subprocess

# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     print(cmd)
#     subprocess.run(cmd, shell=True, check=True)

## Get p-values

### Raw p-vals for each model config given observed/permutted diffs
- 2 layer: diff = r2_T - r2_S
- 3 layer: diff = r2_T - r2_Q

In [ ]:
base_imp_path = BASE_PATH / "perm_res" / "r2_tables"
# Dict of the form {perm_name: df} for each of the 4 types of dfs (parent_2, self_2, parent_3, self_3)
parent_2_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "parent_2.parquet")
    for perm_dir in base_imp_path.iterdir()
}
self_2_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "self_2.parquet")
    for perm_dir in base_imp_path.iterdir()
}
parent_3_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "parent_3.parquet")
    for perm_dir in base_imp_path.iterdir()
}
self_3_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "self_3.parquet")
    for perm_dir in base_imp_path.iterdir()
}

In [ ]:
p_parent_2 = get_p_df(
    search_dict=search_dict, num_layers=2, r2_df_dict=parent_2_df_dict
)

In [ ]:
p_parent_2

In [ ]:
p_parent_2 = get_p_df(
    search_dict=search_dict, num_layers=2, r2_df_dict=parent_2_df_dict
)
p_parent_3 = get_p_df(
    search_dict=search_dict, num_layers=3, r2_df_dict=parent_3_df_dict
)
p_self_2 = get_p_df(search_dict=search_dict, num_layers=2, r2_df_dict=self_2_df_dict)
p_self_3 = get_p_df(search_dict=search_dict, num_layers=3, r2_df_dict=self_3_df_dict)

In [ ]:
comprehensive_dir.mkdir(exist_ok=True, parents=True)
p_parent_2.to_parquet(comprehensive_dir / "parent_2.parquet")
p_parent_3.to_parquet(comprehensive_dir / "parent_3.parquet")
p_self_2.to_parquet(comprehensive_dir / "self_2.parquet")
p_self_3.to_parquet(comprehensive_dir / "self_3.parquet")

#### Look at raw p-vals w/ distribution of differences from permutations for individual model configurations

In [ ]:
def get_sub_df(df, condition_dict):
    mask = pd.Series(True, index=df.index)
    for col, val in condition_dict.items():
        mask &= df[col] == val
    return df[mask]


def build_distribution(minuend, subtrahend, model_config, loop_dir, num_layers, impact):
    diff_list = []
    baseline = None
    for perm_dir in loop_dir.iterdir():
        r2_df = pd.read_parquet(perm_dir / f"{impact}_{num_layers}.parquet")
        display(r2_df)
        cur_row = get_sub_df(r2_df, model_config)
        diff = cur_row[minuend].values[0] - cur_row[subtrahend].values[0]
        diff_list.append(diff)
        if perm_dir.name == "baseline":
            baseline = diff
    return baseline, diff_list

In [ ]:
p_parent_2 = pd.read_parquet(comprehensive_dir / "parent_2.parquet")
p_parent_3 = pd.read_parquet(comprehensive_dir / "parent_3.parquet")
p_self_2 = pd.read_parquet(comprehensive_dir / "self_2.parquet")
p_self_3 = pd.read_parquet(comprehensive_dir / "self_3.parquet")

In [ ]:
sig_neg_df = p_parent_3[
    (p_parent_3["baseline_diff"] < 0) & (p_parent_3["p_val"] < 0.05)
]
for i, row in sig_neg_df.iterrows():
    if i != 3:
        continue
    else:
        print(row.to_dict())
        break

In [ ]:
model_config = {
    "agg": "binary",
    "llama_model": "Llama-3.3-8B-Instruct",
    "desc_seg": "augment",
    "alpha": "0.5_5.0",
    # "baseline_diff": -0.1226,
    # "p_val": 0.000999000999000999,
}
loop_dir = BASE_PATH / "perm_res" / "r2_tables"
num_layers = 3
impact = "parent"
baseline, dist = build_distribution(
    minuend="r2_T",
    subtrahend="r2_S",
    model_config=model_config,
    loop_dir=loop_dir,
    num_layers=num_layers,
    impact=impact,
)
print(f"Baseline: {baseline:.3f}")
plt.hist(dist, bins=30)
plt.show()

### Null distribution of best models across permutations

In [ ]:
def get_global_null(impact, num_layers, loop_dir):
    """
    Populate global null distribution with best r2 score from each permutation (including baseline).

    If two layers, test stat is r2_T - r2_S. If three layers, test stat is r2_T - r2_Q.
    """
    global_null = []
    for perm_dir in loop_dir.iterdir():
        r2_df = pd.read_parquet(perm_dir / f"{impact}_{num_layers}.parquet")
        best_score_this_perm = -99
        for row in r2_df.itertuples():
            if num_layers == 2:
                r2_diff_score = row.r2_T - row.r2_S
            else:
                r2_diff_score = row.r2_T - row.r2_Q
            if r2_diff_score > best_score_this_perm:
                best_score_this_perm = r2_diff_score
        if perm_dir.name == "baseline":
            baseline = best_score_this_perm
            best_row = row
        global_null.append(best_score_this_perm)
    p_val = len([1 for score in global_null if score >= baseline]) / len(global_null)
    return baseline, p_val, global_null, best_row

In [ ]:
loop_dir = BASE_PATH / "perm_res" / "r2_tables"
parent_3_baseline, parent_3_p, parent_3_null, best_parent_row = get_global_null(
    impact="parent", num_layers=3, loop_dir=loop_dir
)
print(f"Baseline: {parent_3_baseline:.3f}")
print(f"P-Value: {parent_3_p:.3f}")
plt.hist(parent_3_null, bins=30)
plt.show()

In [ ]:
loop_dir = BASE_PATH / "perm_res" / "r2_tables"
parent_2_baseline, parent_2_p, parent_2_null = get_global_null(
    impact="parent", num_layers=2, loop_dir=loop_dir
)
print(f"Baseline: {parent_2_baseline:.3f}")
print(f"P-Value: {parent_2_p:.3f}")
plt.hist(parent_2_null, bins=30)
plt.show()

In [ ]:
loop_dir = BASE_PATH / "perm_res" / "r2_tables"
self_2_baseline, self_2_p, self_2_null = get_global_null(
    impact="self", num_layers=2, loop_dir=loop_dir
)
print(f"Baseline: {self_2_baseline:.3f}")
print(f"P-Value: {self_2_p:.3f}")
plt.hist(self_2_null, bins=30)
plt.show()

In [ ]:
loop_dir = BASE_PATH / "perm_res" / "r2_tables"
self_3_baseline, self_3_p, self_3_null = get_global_null(
    impact="self", num_layers=3, loop_dir=loop_dir
)
print(f"Baseline: {self_3_baseline:.3f}")
print(f"P-Value: {self_3_p:.3f}")
plt.hist(self_3_null, bins=30)
plt.show()

### Conjuction p-vals

1) Single diff for each model (minimum across hypotheses)
    - Will have to duplicate 2-layer model configs to match 3-model configs

2) Null distribution of diffs (one per permutation)
    - Start w/ 3-dimensional data
        - Axis1: permutations
        - Axis2: model config
        - Axis3: Hypotheses

    - Collapse axis3 by taking minimum diff across all hypotheses
    - Collapse axis2 by taking maximum diff across all model configs
    - End with list of 1001 diffs (including baseline)

In [ ]:
p_parent_2 = pd.read_parquet(comprehensive_dir / "parent_2.parquet")
p_parent_3 = pd.read_parquet(comprehensive_dir / "parent_3.parquet")
p_self_2 = pd.read_parquet(comprehensive_dir / "self_2.parquet")
p_self_3 = pd.read_parquet(comprehensive_dir / "self_3.parquet")

In [ ]:
p_parent_2_dup = expand_alpha_rows(p_parent_2)
p_self_2_dup = expand_alpha_rows(p_self_2)

Single diff per model

In [ ]:
parent_combined = collapse_hypothesis(p_parent_2_dup, p_parent_3)
self_combined = collapse_hypothesis(p_self_2_dup, p_self_3)

#### Null dist shared across models

In [ ]:
null_dist_parent, parent_baseline = get_collapsed_null(impact_type="parent")
null_dist_self, self_baseline = get_collapsed_null(impact_type="self")

In [ ]:
parent_baseline.head()

In [ ]:
import numpy as np

plt.hist(parent_baseline["min_diff"], bins=50)

intervals = np.arange(0.0, 0.05, 0.01)
plt.xticks(intervals)
plt.show()

In [ ]:
plt.hist(null_dist_parent, bins=50)
plt.title("Parent Impact Null Distribution (1000 perms)")
plt.xlabel("Difference in R²")
plt.ylabel("Frequency")
plt.show()

In [ ]:
import numpy as np

plt.hist(self_baseline["min_diff"], bins=50)

intervals = np.arange(0, 0.04, 0.01)
plt.xticks(intervals)
plt.show()

In [ ]:
plt.hist(null_dist_self, bins=50)
plt.title("Self Impact Null Distribution (1000 perms)")
plt.xlabel("Difference in R²")
plt.ylabel("Frequency")
plt.show()

#### Get p-values

- share nulls across all models

In [ ]:
pd.read_parquet(
    "/data/MLDSST/mataray/DAWBA/perm_res/preds/perm_104/parent/pred_df_combined.parquet"
)

In [ ]:
parent_baseline["p_val"] = parent_baseline["min_diff"].apply(
    lambda baseline: get_ranking(baseline, null_dist_parent)
)
parent_combined_sig = parent_baseline[
    (parent_baseline["p_val"] < 0.05) & (parent_baseline["neg_r2"] != 1)
].sort_values("min_diff", ascending=False)
print(f"Significant model configs: {len(parent_combined_sig)} / {len(parent_baseline)}")
assert parent_combined_sig["neg_r2"].sum() == 0
parent_combined_sig.sort_values("min_diff", ascending=False)

In [ ]:
self_baseline["p_val"] = self_baseline["min_diff"].apply(
    lambda baseline: get_ranking(baseline, null_dist_self)
)
self_combined_sig = self_baseline[
    (self_baseline["p_val"] < 0.05) & (self_baseline["neg_r2"] != 1)
].sort_values("min_diff", ascending=False)
print(f"Significant model configs: {len(self_combined_sig)} / {len(self_baseline)}")
assert self_combined_sig["neg_r2"].sum() == 0
self_combined_sig.sort_values("min_diff", ascending=False)

In [ ]:
all_combined_sig = parent_combined_sig.merge(
    self_combined_sig,
    how="inner",
    on=["agg", "llama_model", "desc_seg", "alpha"],
    suffixes=["_p", "_s"],
)
all_combined_sig.sort_values("min_diff_p")

## Hypothesis 3--> 2T-3Q>0

In [ ]:
from shutil import rmtree

r2_export_dir = BASE_PATH / "perm_res/switch/r2_dfs"
perm_ids = [f"{i:03d}" for i in range(1, 1000)]

### Create r2 tables

In [ ]:
parent_permed = pd.read_parquet(BASE_PATH / "perm_res/switch/perm_diffs/parent.parquet")
self_permed = pd.read_parquet(BASE_PATH / "perm_res/switch/perm_diffs/self.parquet")

In [ ]:
def compute_r2s(g):
    scores = {
        "r2_2T": r2_score(g["true"], g["pred_2T"]),
        "r2_3Q": r2_score(g["true"], g["pred_3Q"]),
    }
    for pid in perm_ids:
        scores[f"r2_2T_perm_{pid}"] = r2_score(g["true"], g[f"pred_2T_perm_{pid}"])
        scores[f"r2_3Q_perm_{pid}"] = r2_score(g["true"], g[f"pred_3Q_perm_{pid}"])
    return pd.Series(scores)


def get_r2_df(perm_df):
    return (
        perm_df.groupby(["agg_type", "llama_model", "desc_seg", "alpha", "repeat"])
        .apply(compute_r2s)
        .reset_index()
        .groupby(["agg_type", "llama_model", "desc_seg", "alpha"])[
            ["r2_2T", "r2_3Q"]
            + [f"r2_2T_perm_{pid}" for pid in perm_ids]
            + [f"r2_3Q_perm_{pid}" for pid in perm_ids]
        ]
        .mean()
        .reset_index()
    )

In [ ]:
parent_r2_df = get_r2_df(parent_permed)
self_r2_df = get_r2_df(self_permed)

In [ ]:
if r2_export_dir.exists():
    rmtree(r2_export_dir)
r2_export_dir.mkdir(exist_ok=False, parents=True)
parent_r2_df.to_parquet(r2_export_dir / "parent.parquet")
self_r2_df.to_parquet(r2_export_dir / "self.parquet")

### Analyze tables

In [ ]:
parent_r2_df = pd.read_parquet(r2_export_dir / "parent.parquet")
self_r2_df = pd.read_parquet(r2_export_dir / "self.parquet")

In [ ]:
parent_r2_df

In [ ]:
def get_null_dist(r2_df):
    df_sub = r2_df[["agg_type", "llama_model", "desc_seg", "alpha"]].copy()
    null_diffs = []
    for perm_id in ["baseline"] + perm_ids:
        suffix = "" if perm_id == "baseline" else f"_perm_{perm_id}"
        diff_list = r2_df[f"r2_2T{suffix}"] - r2_df[f"r2_3Q{suffix}"]
        if perm_id == "baseline":
            df_sub["diff_baseline"] = diff_list
            # df_sub['neg_r2'] = (r2_df[f'r2_2T'] < 0).astype(int)
            # df_sub['neg_r2'] = (r2_df[f'r2_3Q'] < 0).astype(int)
        max_diff = max(diff_list, key=abs)
        null_diffs.append(max_diff)
    return df_sub, null_diffs


baseline_parent, null_parent = get_null_dist(parent_r2_df)
baseline_self, null_self = get_null_dist(self_r2_df)

In [ ]:
plt.hist(baseline_parent["diff_baseline"], bins=30)
plt.show()

In [ ]:
plt.hist(null_parent, bins=30)
plt.show()

In [ ]:
plt.hist(baseline_self["diff_baseline"], bins=30)
plt.show()

In [ ]:
plt.hist(null_self, bins=30)
plt.show()

In [ ]:
def get_two_tail_ranking(baseline, null_dist):
    extreme = len(
        [
            1
            for null_diff in null_dist
            if ((null_diff >= abs(baseline)) or (null_diff <= (-1 * abs(baseline))))
        ]
    )
    p_val = extreme / len(null_dist)
    return p_val

In [ ]:
baseline_parent["p_val"] = baseline_parent["diff_baseline"].apply(
    lambda baseline: get_two_tail_ranking(baseline, null_parent)
)
parent_combined_sig = baseline_parent[(baseline_parent["p_val"] <= 0.05)].sort_values(
    "diff_baseline", ascending=False
)
print(f"Significant model configs: {len(parent_combined_sig)} / {len(baseline_parent)}")
# assert parent_combined_sig['neg_r2'].sum() == 0
parent_combined_sig.sort_values("diff_baseline", ascending=False)

In [ ]:
baseline_self["p_val"] = baseline_self["diff_baseline"].apply(
    lambda baseline: get_two_tail_ranking(baseline, null_self)
)
self_combined_sig = baseline_self[(baseline_self["p_val"] < 0.2)].sort_values(
    "diff_baseline", ascending=False
)
print(f"Significant model configs: {len(self_combined_sig)} / {len(baseline_self)}")
# assert self_combined_sig['neg_r2'].sum() == 0
self_combined_sig.sort_values("diff_baseline", ascending=False)

# Part 2

In [ ]:
## AVG ABSOLUTE ERROR
def get_cohens_t_df(df, num_layers, num_repeats=10):
    df_layered = df[df["num_layers"] == num_layers].copy()
    reduced_col = "pred_stage1" if num_layers == 2 else "pred_stage2_total"

    # Diff in abs error
    reduced_error = np.abs(df_layered["true"] - df_layered[reduced_col])
    full_error = np.abs(df_layered["true"] - df_layered["pred_stage3_total"])
    df_layered["diff_abs_error"] = reduced_error - full_error
    # Avg + stdev of diff for each model
    result = (
        df_layered.groupby(
            ["num_layers", "agg_type", "llama_model", "desc_seg", "alpha"]
        )["diff_abs_error"]
        .agg(avg_diff="mean", std_diff="std", n="count")
        .reset_index()
    )

    result["n"] = result["n"] / num_repeats  # 10 repeats
    assert result["n"].nunique() == 1, f"Unique: {result["n"].unique()}"
    result["cohens_d"] = result["avg_diff"] / result["std_diff"]
    result["t_stat"] = result["cohens_d"] * np.sqrt(result["n"])
    # result["cohens_d"]
    return result

In [ ]:
baseline_dir = BASE_PATH / "perm_res/preds/baseline"
# 71 total
parent_baseline_df = pd.read_parquet(baseline_dir / "parent/pred_df_combined.parquet")
# 103 total
self_baseline_df = pd.read_parquet(baseline_dir / "self/pred_df_combined.parquet")

Cohens D

- avg(diff) / stdev(diff) for each model

In [ ]:
cohen_parent_2 = get_cohens_t_df(parent_baseline_df, 2)
cohen_parent_3 = get_cohens_t_df(parent_baseline_df, 3)
cohen_self_2 = get_cohens_t_df(self_baseline_df, 2)
cohen_self_3 = get_cohens_t_df(self_baseline_df, 3)

In [ ]:
cohen_parent_3.sort_values("cohens_d", ascending=False)

In [ ]:
cohen_self_3.sort_values("cohens_d", ascending=False)

## P-vals w/ T-stat

In [ ]:
def collapse_hypothesis_w_t(df1, df2):
    """
    Collapses hypothesis across 2 and 3 layers by taking the min of the baseline diffs.
    Returns a df with columns: agg, llama_model, desc_seg, alpha, min_diff
    """
    combined = pd.merge(
        df1,
        df2,
        on=["agg_type", "llama_model", "desc_seg", "alpha"],
        suffixes=("_2", "_3"),
    )

    combined["min_t_stat"] = combined.apply(
        lambda row: min(row.t_stat_2, row.t_stat_3), axis=1
    )
    return combined.sort_values("min_t_stat", ascending=False)


def get_collapsed_null_t(
    impact_type, perm_dir_base=BASE_PATH / "perm_res" / "preds", single_repeat=False
):
    """
    Creates null distribution by first collapsing hypothesis axis (min) then collapsing model axis (max)
    for each permutation. Results in a single diff value per permutation to create a shared null distribution
    for a given impact_type.

    Params
    -----
    impact_type: str
        One of ['self', 'parent']
    """
    null_dist = []
    baseline_df = None
    for perm_dir in perm_dir_base.iterdir():
        pred_df = pd.read_parquet(perm_dir / impact_type / "pred_df_combined.parquet")
        if single_repeat:
            pred_df = pred_df[pred_df["repeat"] == 0]
        # Add cohen_d + t_stat
        num_repeats = 1 if single_repeat else 10
        cohen_t_2 = get_cohens_t_df(pred_df, 2, num_repeats=num_repeats)
        cohen_t_3 = get_cohens_t_df(pred_df, 3, num_repeats=num_repeats)
        # Duplicate 2 layer to match 3 layer alphas
        cohen_t_2_dup = expand_alpha_rows(cohen_t_2)
        # Collpase hypothesis axis
        # t stat of 2 layer df is for hypotheis 2T > 2S
        # t stat of 3 layer df is for hypotheis 3T > 3Q
        combined = collapse_hypothesis_w_t(cohen_t_2_dup, cohen_t_3)
        if perm_dir.name == "baseline":
            baseline_df = combined
        max_t = max(combined["min_t_stat"])
        null_dist.append(max_t)
    return null_dist, baseline_df

### Global

In [ ]:
# cohen_parent_2_dup = expand_alpha_rows(cohen_parent_2)
# cohen_self_2_dup = expand_alpha_rows(cohen_self_2)
# parent_combined = collapse_hypothesis_w_t(cohen_parent_2_dup, cohen_parent_3)
# self_combined = collapse_hypothesis_w_t(cohen_self_2_dup, cohen_self_3)

In [ ]:
null_dist_parent, parent_baseline = get_collapsed_null_t(impact_type="parent")
null_dist_self, self_baseline = get_collapsed_null_t(impact_type="self")

p-vals

In [ ]:
parent_baseline["p_val"] = parent_baseline["min_t_stat"].apply(
    lambda baseline: get_ranking(baseline, null_dist_parent)
)
parent_combined_sig = parent_baseline[parent_baseline["p_val"] < 0.05].sort_values(
    "min_t_stat", ascending=False
)
print(f"Significant model configs: {len(parent_combined_sig)} / {len(parent_baseline)}")
parent_combined_sig

In [ ]:
self_baseline["p_val"] = self_baseline["min_t_stat"].apply(
    lambda baseline: get_ranking(baseline, null_dist_self)
)
self_combined_sig = self_baseline[self_baseline["p_val"] < 0.05].sort_values(
    "min_t_stat", ascending=False
)
print(f"Significant model configs: {len(self_combined_sig)} / {len(self_baseline)}")
self_combined_sig

### With single repeat

In [ ]:
null_dist_parent_single, parent_baseline_single = get_collapsed_null_t(
    impact_type="parent", single_repeat=True
)
null_dist_self_single, self_baseline_single = get_collapsed_null_t(
    impact_type="self", single_repeat=True
)

In [ ]:
parent_baseline_single["p_val"] = parent_baseline_single["min_t_stat"].apply(
    lambda baseline: get_ranking(baseline, null_dist_parent_single)
)
parent_combined_sig_single = parent_baseline_single[
    parent_baseline_single["p_val"] < 0.05
].sort_values("min_t_stat", ascending=False)
print(
    f"Significant model configs: {len(parent_combined_sig_single)} / {len(parent_baseline_single)}"
)
parent_combined_sig_single

In [ ]:
self_baseline_single["p_val"] = self_baseline_single["min_t_stat"].apply(
    lambda baseline: get_ranking(baseline, null_dist_self_single)
)
self_combined_sig_single = self_baseline_single[
    self_baseline_single["p_val"] < 0.3
].sort_values("min_t_stat", ascending=False)
print(
    f"Significant model configs: {len(self_combined_sig_single)} / {len(self_baseline_single)}"
)
self_combined_sig_single